In [1]:
# CELL 1 — Install moviepy (ffmpeg is pre-installed on Kaggle)
import subprocess
subprocess.run(['pip', 'install', 'moviepy==1.0.3', '--quiet'], check=True)
print('Dependencies installed.')

Dependencies installed.


In [2]:
# CELL 2 — Imports and paths
import os, re, io, warnings, subprocess
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

warnings.filterwarnings('ignore', category=SyntaxWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# ── File path placeholders ─────────────────────────────────────────────────
VIDEO_PATH        = '/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs PAK.mp4'
OUTPUT_DIR        = '/kaggle/working'
CLIPS_DIR         = os.path.join(OUTPUT_DIR, 'clips')

CSV_30S           = os.path.join(OUTPUT_DIR, 'intensity_30s.csv')
CSV_SORTED        = os.path.join(OUTPUT_DIR, 'intensity_sorted.csv')
CSV_GROUPED       = os.path.join(OUTPUT_DIR, 'peaks_grouped.csv')
CSV_TOP_CHRONO    = os.path.join(OUTPUT_DIR, 'top10_chronological.csv')
FINAL_VIDEO       = os.path.join(OUTPUT_DIR, 'highlights_final.mp4')

os.makedirs(CLIPS_DIR, exist_ok=True)

# ── Tunable parameters ─────────────────────────────────────────────────────
SAMPLE_INTERVAL   = 30           # seconds between samples
GROUP_WINDOW      = 120          # seconds — peaks within ±120s are same group
TOP_N             = 10           # how many peaks to include in highlights
CLIP_START_OFFSET = 60           # clip starts this many sec BEFORE peak
CLIP_END_OFFSET   = 30           # clip ends this many sec BEFORE peak

print(f'Video input : {VIDEO_PATH}')
print(f'Video exists: {os.path.exists(VIDEO_PATH)}')
print(f'Output dir  : {OUTPUT_DIR}')

Video input : /kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs PAK.mp4
Video exists: True
Output dir  : /kaggle/working


In [3]:
# CELL 3 — The raw SVG path data (paste verbatim from browser inspect)
SVG_PATH_D = """M 0.0,100.0 C 1.0,87.0 2.0,39.3 5.0,34.8 C 8.0,30.3 11.0,70.8 15.0,77.5 C 19.0,84.3 21.0,70.2 25.0,68.4 C 29.0,66.5 31.0,67.9 35.0,68.3 C 39.0,68.7 41.0,71.4 45.0,70.6 C 49.0,69.8 51.0,63.9 55.0,64.2 C 59.0,64.4 61.0,67.8 65.0,71.8 C 69.0,75.7 71.0,85.2 75.0,83.9 C 79.0,82.7 81.0,66.1 85.0,65.5 C 89.0,64.8 91.0,81.7 95.0,80.8 C 99.0,79.8 101.0,64.2 105.0,60.7 C 109.0,57.2 111.0,60.2 115.0,63.2 C 119.0,66.2 121.0,72.9 125.0,75.6 C 129.0,78.4 131.0,77.1 135.0,76.9 C 139.0,76.8 141.0,78.5 145.0,75.0 C 149.0,71.5 151.0,58.6 155.0,59.5 C 159.0,60.5 161.0,91.6 165.0,79.7 C 169.0,67.8 171.0,0.5 175.0,0.0 C 179.0,-0.5 181.0,63.4 185.0,77.4 C 189.0,91.4 191.0,71.0 195.0,70.2 C 199.0,69.4 201.0,75.9 205.0,73.2 C 209.0,70.5 211.0,58.8 215.0,56.8 C 219.0,54.9 221.0,63.8 225.0,63.4 C 229.0,63.0 231.0,54.1 235.0,54.8 C 239.0,55.4 241.0,59.5 245.0,66.5 C 249.0,73.6 251.0,91.2 255.0,90.0 C 259.0,88.8 261.0,61.5 265.0,60.5 C 269.0,59.6 271.0,83.6 275.0,85.4 C 279.0,87.3 281.0,69.7 285.0,69.8 C 289.0,69.9 291.0,83.5 295.0,86.1 C 299.0,88.6 301.0,84.0 305.0,82.5 C 309.0,80.9 311.0,81.1 315.0,78.4 C 319.0,75.6 321.0,67.3 325.0,68.7 C 329.0,70.2 331.0,82.8 335.0,85.6 C 339.0,88.4 341.0,81.9 345.0,82.8 C 349.0,83.6 351.0,88.6 355.0,90.0 C 359.0,91.4 361.0,91.5 365.0,90.0 C 369.0,88.5 371.0,87.5 375.0,82.3 C 379.0,77.1 381.0,62.5 385.0,64.0 C 389.0,65.6 391.0,88.4 395.0,90.0 C 399.0,91.6 401.0,73.4 405.0,71.9 C 409.0,70.4 411.0,85.5 415.0,82.4 C 419.0,79.4 421.0,57.1 425.0,56.6 C 429.0,56.2 431.0,76.0 435.0,80.0 C 439.0,84.1 441.0,76.4 445.0,76.9 C 449.0,77.4 451.0,82.3 455.0,82.6 C 459.0,82.8 461.0,76.6 465.0,78.1 C 469.0,79.6 471.0,87.7 475.0,90.0 C 479.0,92.3 481.0,89.4 485.0,89.4 C 489.0,89.4 491.0,92.4 495.0,90.0 C 499.0,87.6 501.0,78.8 505.0,77.3 C 509.0,75.8 511.0,83.9 515.0,82.7 C 519.0,81.4 521.0,72.0 525.0,71.3 C 529.0,70.5 531.0,76.2 535.0,79.0 C 539.0,81.9 541.0,88.6 545.0,85.4 C 549.0,82.2 551.0,63.0 555.0,63.0 C 559.0,63.0 561.0,80.0 565.0,85.3 C 569.0,90.7 571.0,89.1 575.0,90.0 C 579.0,90.9 581.0,90.0 585.0,90.0 C 589.0,90.0 591.0,92.6 595.0,90.0 C 599.0,87.4 601.0,81.0 605.0,76.8 C 609.0,72.6 611.0,66.8 615.0,68.8 C 619.0,70.9 621.0,84.4 625.0,87.0 C 629.0,89.5 631.0,81.8 635.0,81.4 C 639.0,81.1 641.0,85.0 645.0,85.3 C 649.0,85.5 651.0,84.8 655.0,82.8 C 659.0,80.9 661.0,76.5 665.0,75.5 C 669.0,74.6 671.0,86.0 675.0,78.1 C 679.0,70.1 681.0,37.1 685.0,35.8 C 689.0,34.5 691.0,60.9 695.0,71.5 C 699.0,82.1 701.0,85.0 705.0,88.7 C 709.0,92.3 711.0,90.4 715.0,89.7 C 719.0,89.0 721.0,85.0 725.0,85.1 C 729.0,85.2 731.0,90.6 735.0,90.0 C 739.0,89.4 741.0,87.8 745.0,82.0 C 749.0,76.3 751.0,61.2 755.0,61.1 C 759.0,60.9 761.0,75.6 765.0,81.4 C 769.0,87.1 771.0,88.3 775.0,90.0 C 779.0,91.7 781.0,90.0 785.0,90.0 C 789.0,90.0 791.0,90.5 795.0,90.0 C 799.0,89.5 801.0,92.5 805.0,87.3 C 809.0,82.1 811.0,65.0 815.0,64.1 C 819.0,63.2 821.0,77.8 825.0,82.9 C 829.0,87.9 831.0,90.5 835.0,89.4 C 839.0,88.3 841.0,83.1 845.0,77.4 C 849.0,71.7 851.0,58.7 855.0,60.9 C 859.0,63.1 861.0,82.7 865.0,88.5 C 869.0,94.3 871.0,89.7 875.0,90.0 C 879.0,90.3 881.0,90.0 885.0,90.0 C 889.0,90.0 891.0,91.2 895.0,90.0 C 899.0,88.8 901.0,85.9 905.0,84.2 C 909.0,82.4 911.0,80.0 915.0,81.2 C 919.0,82.4 921.0,89.9 925.0,90.0 C 929.0,90.1 931.0,90.7 935.0,81.5 C 939.0,72.4 941.0,51.3 945.0,44.5 C 949.0,37.6 951.0,43.1 955.0,47.4 C 959.0,51.7 961.0,66.1 965.0,66.0 C 969.0,65.8 971.0,44.6 975.0,46.7 C 979.0,48.8 981.0,73.1 985.0,76.5 C 989.0,79.9 992.0,66.3 995.0,63.8 C 998.0,61.2 999.0,56.5 1000.0,63.8 C 1001.0,71.0 1000.0,92.8 1000.0,100.0"""

print(f'SVG path length: {len(SVG_PATH_D)} chars')

SVG path length: 3549 chars


In [4]:
# CELL 4 — Get video duration (needed to map svg x→time)
def get_video_duration(path):
    """Return video duration in seconds using ffprobe."""
    if not os.path.exists(path):
        raise FileNotFoundError(f'Video file not found: {path}')
    result = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'default=noprint_wrappers=1:nokey=1', path],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f'ffprobe failed: {result.stderr}')
    return float(result.stdout.strip())

VIDEO_DURATION = get_video_duration(VIDEO_PATH)
h, m, s = int(VIDEO_DURATION)//3600, (int(VIDEO_DURATION)%3600)//60, int(VIDEO_DURATION)%60
print(f'Video duration: {VIDEO_DURATION:.2f}s  ({h}h {m}m {s}s)')

Video duration: 11255.13s  (3h 7m 35s)


In [5]:
# CELL 5 — Parse SVG path into cubic bezier segments
#
# SVG path commands used by YouTube's heatmap:
#   M x,y        → move to (start point)
#   C x1,y1 x2,y2 x,y  → cubic bezier (ctrl1, ctrl2, end)
#
# We store each segment as (P0, P1_ctrl, P2_ctrl, P3_end).

def parse_svg_path(d_attr):
    """Parse SVG 'd' into list of cubic bezier segments: (p0, p1, p2, p3)."""
    try:
        tokens = re.split(r'\s+', d_attr.strip())
    except Exception as e:
        raise ValueError(f'Malformed SVG path: {e}')
    segments = []
    current = None
    i = 0
    while i < len(tokens):
        t = tokens[i]
        if t == 'M':
            try:
                x, y = map(float, tokens[i+1].split(','))
            except Exception as e:
                raise ValueError(f'Malformed M command at token {i}: {e}')
            current = (x, y)
            i += 2
        elif t == 'C':
            if i + 3 >= len(tokens):
                raise ValueError(f'Truncated C command at token {i}')
            try:
                p1 = tuple(map(float, tokens[i+1].split(',')))
                p2 = tuple(map(float, tokens[i+2].split(',')))
                p3 = tuple(map(float, tokens[i+3].split(',')))
            except Exception as e:
                raise ValueError(f'Malformed C command at token {i}: {e}')
            segments.append((current, p1, p2, p3))
            current = p3
            i += 4
        else:
            i += 1
    if not segments:
        raise ValueError('No bezier segments parsed — path may be empty or malformed')
    return segments

segments = parse_svg_path(SVG_PATH_D)
print(f'Parsed {len(segments)} cubic bezier segments.')
print(f'First segment: p0={segments[0][0]}  p3={segments[0][3]}')
print(f'Last  segment: p0={segments[-1][0]}  p3={segments[-1][3]}')

Parsed 102 cubic bezier segments.
First segment: p0=(0.0, 100.0)  p3=(5.0, 34.8)
Last  segment: p0=(1000.0, 63.8)  p3=(1000.0, 100.0)


In [6]:
# CELL 6 — Evaluate any x along the curve by solving the bezier for t
#
# For a cubic bezier with control points P0..P3, the parametric form is:
#   B(t) = (1-t)^3 P0 + 3(1-t)^2 t P1 + 3(1-t) t^2 P2 + t^3 P3
# Since x(t) is monotonic in each segment (x values of anchors are increasing),
# we can binary-search for t such that x(t) = target_x.

def bezier_point(seg, t):
    p0, p1, p2, p3 = seg
    mt = 1 - t
    x = mt**3*p0[0] + 3*mt**2*t*p1[0] + 3*mt*t**2*p2[0] + t**3*p3[0]
    y = mt**3*p0[1] + 3*mt**2*t*p1[1] + 3*mt*t**2*p2[1] + t**3*p3[1]
    return x, y

def y_at_x(target_x, segs):
    """Return y-value of the curve at target_x (x in SVG space, 0..1000)."""
    for seg in segs:
        if seg[0][0] <= target_x <= seg[3][0] + 1e-9:
            lo, hi = 0.0, 1.0
            for _ in range(60):    # 60 iters = sub-pixel precision
                mid = (lo + hi) / 2
                x_mid, _ = bezier_point(seg, mid)
                if x_mid < target_x:
                    lo = mid
                else:
                    hi = mid
            _, y = bezier_point(seg, (lo+hi)/2)
            return y
    # fallback: clamp to nearest endpoint
    if target_x <= segs[0][0][0]:
        return segs[0][0][1]
    return segs[-1][3][1]

# quick sanity: evaluate at x=0, 500, 1000
for x in [0, 500, 1000]:
    print(f'y at x={x}: {y_at_x(x, segments):.2f}')

y at x=0: 100.00
y at x=500: 83.31
y at x=1000: 63.80


In [7]:
# CELL 7 — Step 1 & 2: build dataframe at 30-second intervals
#
# Mapping:
#   SVG x ∈ [0, 1000]  ↔  video time ∈ [0, VIDEO_DURATION]
#   SVG y ∈ [0, 100]   ↔  intensity = 100 − y, clamped to [0, 100]

def secs_to_ts(s):
    s = int(round(s))
    return f'{s//3600}:{(s%3600)//60:02d}:{s%60:02d}'

rows = []
t = 0
while t <= VIDEO_DURATION:
    x = (t / VIDEO_DURATION) * 1000.0
    y = y_at_x(x, segments)
    intensity = max(0.0, min(100.0, 100.0 - y))
    rows.append({
        'timestamp': secs_to_ts(t),
        'seconds':   int(t),
        'intensity': round(intensity, 2),
    })
    t += SAMPLE_INTERVAL

df_30s = pd.DataFrame(rows)
df_30s.to_csv(CSV_30S, index=False)
print(f'Saved: {CSV_30S}  ({len(df_30s)} rows)')
df_30s.head(10)

Saved: /kaggle/working/intensity_30s.csv  (376 rows)


,timestamp,seconds,intensity
0,0:00:00,0,0.00
1,0:00:30,30,50.12
2,0:01:00,60,65.52
3,0:01:30,90,57.84
4,0:02:00,120,41.43
5,0:02:30,150,27.28
6,0:03:00,180,21.19
7,0:03:30,210,21.64
8,0:04:00,240,26.53
9,0:04:30,270,30.86


In [8]:
# CELL 8 — Step 3: sort descending by intensity
df_sorted = df_30s.sort_values(
    by=['intensity', 'seconds'],
    ascending=[False, True]         # tie-break: earlier timestamp wins
).reset_index(drop=True)
df_sorted.to_csv(CSV_SORTED, index=False)
print(f'Saved: {CSV_SORTED}')
df_sorted.head(15)

Saved: /kaggle/working/intensity_sorted.csv


,timestamp,seconds,intensity
0,0:33:00,1980,98.95
1,0:32:30,1950,95.33
2,0:33:30,2010,82.26
3,0:32:00,1920,71.52
4,0:01:00,60,65.52
5,2:08:30,7710,64.21
6,2:09:00,7740,60.66
7,2:58:00,10680,58.74
8,2:08:00,7680,58.35
9,0:01:30,90,57.84


In [9]:
# CELL 9 — Step 4: group peaks within 2 minutes
#
# Greedy grouping: walk the sorted-by-intensity list. For each unclaimed row,
# start a new group (it becomes the group representative — highest intensity
# of any member). Absorb all other unclaimed rows within ±GROUP_WINDOW sec.
# Because we process in descending intensity order with earlier-timestamp
# tie-break, the group rep is guaranteed to be the correct winner.

used   = set()
groups = []

for _, row in df_sorted.iterrows():
    if row['seconds'] in used:
        continue
    # this row becomes the group representative
    rep = row
    used.add(int(row['seconds']))
    # absorb nearby unclaimed rows
    for _, other in df_sorted.iterrows():
        if other['seconds'] in used:
            continue
        if abs(other['seconds'] - rep['seconds']) <= GROUP_WINDOW:
            used.add(int(other['seconds']))
    groups.append({
        'group_id':            len(groups) + 1,
        'selected_timestamp':  rep['timestamp'],
        'selected_seconds':    int(rep['seconds']),
        'selected_intensity':  float(rep['intensity']),
    })

df_grouped = pd.DataFrame(groups)
df_grouped.to_csv(CSV_GROUPED, index=False)
print(f'Saved: {CSV_GROUPED}  ({len(df_grouped)} groups)')
df_grouped.head(15)

Saved: /kaggle/working/peaks_grouped.csv  (61 groups)


,group_id,selected_timestamp,selected_seconds,selected_intensity
0,1,0:33:00,1980,98.95
1,2,0:01:00,60,65.52
2,3,2:08:30,7710,64.21
3,4,2:58:00,10680,58.74
4,5,3:03:00,10980,52.86
5,6,0:44:00,2640,45.24
6,7,0:40:30,2430,43.46
7,8,1:20:00,4800,42.61
8,9,0:20:30,1230,40.75
9,10,0:29:00,1740,40.54


In [10]:
# CELL 10 — Step 5 setup: select top 10, save chronological order
df_top = df_grouped.head(TOP_N).copy()
df_top_chrono = df_top.sort_values('selected_seconds').reset_index(drop=True)
df_top_chrono.to_csv(CSV_TOP_CHRONO, index=False)
print(f'Top {TOP_N} peaks (chronological):')
print()
print(df_top_chrono.to_string(index=False))
print()
print(f'Saved: {CSV_TOP_CHRONO}')

Top 10 peaks (chronological):

 group_id selected_timestamp  selected_seconds  selected_intensity
        2            0:01:00                60               65.52
        9            0:20:30              1230               40.75
       10            0:29:00              1740               40.54
        1            0:33:00              1980               98.95
        7            0:40:30              2430               43.46
        6            0:44:00              2640               45.24
        8            1:20:00              4800               42.61
        3            2:08:30              7710               64.21
        4            2:58:00             10680               58.74
        5            3:03:00             10980               52.86

Saved: /kaggle/working/top10_chronological.csv


In [11]:
# CELL 11 — Step 5: extract 30-second clips for each top peak
#
# Clip window for each peak at time T:
#     [max(0, T − 60),  min(DURATION, T − 30)]
# Re-encode (libx264) for clean frame-accurate cuts.

def fmt(s): return f'{int(s)//3600:02d}:{(int(s)%3600)//60:02d}:{int(s)%60:02d}'

clip_paths = []

for _, row in df_top_chrono.iterrows():
    peak_sec = int(row['selected_seconds'])
    start    = max(0, peak_sec - CLIP_START_OFFSET)
    end      = min(VIDEO_DURATION, peak_sec - CLIP_END_OFFSET)
    duration = end - start

    if duration <= 0:
        print(f"  SKIP peak {row['selected_timestamp']}: invalid clip window")
        continue

    safe_ts   = row['selected_timestamp'].replace(':', '-')
    clip_file = os.path.join(CLIPS_DIR, f"clip_{int(row['group_id']):02d}_{safe_ts}.mp4")

    print(f"  [{int(row['group_id']):02d}] peak {row['selected_timestamp']} "
          f"({row['selected_intensity']:.1f}%)  →  clip {fmt(start)} – {fmt(end)}")

    cmd = [
        'ffmpeg', '-y',
        '-ss', str(start),
        '-i', VIDEO_PATH,
        '-t', str(duration),
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
        '-c:a', 'aac', '-b:a', '128k',
        '-avoid_negative_ts', 'make_zero',
        '-loglevel', 'error',
        clip_file
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f"       FAILED:\n{proc.stderr[-400:]}")
    else:
        clip_paths.append(clip_file)
        print(f"       saved ({os.path.getsize(clip_file)/1024:.0f} KB)")

print(f'\nExtracted {len(clip_paths)} / {len(df_top_chrono)} clips.')

  [02] peak 0:01:00 (65.5%)  →  clip 00:00:00 – 00:00:30
       saved (5262 KB)
  [09] peak 0:20:30 (40.8%)  →  clip 00:19:30 – 00:20:00
       saved (8077 KB)
  [10] peak 0:29:00 (40.5%)  →  clip 00:28:00 – 00:28:30
       saved (5397 KB)
  [01] peak 0:33:00 (99.0%)  →  clip 00:32:00 – 00:32:30
       saved (4976 KB)
  [07] peak 0:40:30 (43.5%)  →  clip 00:39:30 – 00:40:00
       saved (6667 KB)
  [06] peak 0:44:00 (45.2%)  →  clip 00:43:00 – 00:43:30
       saved (5337 KB)
  [08] peak 1:20:00 (42.6%)  →  clip 01:19:00 – 01:19:30
       saved (5278 KB)
  [03] peak 2:08:30 (64.2%)  →  clip 02:07:30 – 02:08:00
       saved (6810 KB)
  [04] peak 2:58:00 (58.7%)  →  clip 02:57:00 – 02:57:30
       saved (4728 KB)
  [05] peak 3:03:00 (52.9%)  →  clip 03:02:00 – 03:02:30
       saved (4733 KB)

Extracted 10 / 10 clips.


In [12]:
# CELL 12 — Step 6: merge clips in chronological order
if not clip_paths:
    raise RuntimeError('No clips available to merge — check Cell 11 output.')

concat_list_path = os.path.join(OUTPUT_DIR, 'concat_list.txt')
with open(concat_list_path, 'w') as f:
    for cp in clip_paths:
        f.write(f"file '{cp}'\n")

print(f'Merging {len(clip_paths)} clips → {FINAL_VIDEO}\n')

proc = subprocess.run([
    'ffmpeg', '-y',
    '-f', 'concat', '-safe', '0',
    '-i', concat_list_path,
    '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
    '-c:a', 'aac', '-b:a', '128k',
    '-loglevel', 'error',
    FINAL_VIDEO
], capture_output=True, text=True)

if proc.returncode != 0:
    print('MERGE FAILED:')
    print(proc.stderr)
else:
    size_mb = os.path.getsize(FINAL_VIDEO) / 1_048_576
    total_s = len(clip_paths) * (CLIP_START_OFFSET - CLIP_END_OFFSET)
    print('SUCCESS')
    print(f'  File     : {FINAL_VIDEO}')
    print(f'  Size     : {size_mb:.1f} MB')
    print(f'  Runtime  : ~{total_s//60}m {total_s%60}s  ({len(clip_paths)} × 30-sec clips)')

Merging 10 clips → /kaggle/working/highlights_final.mp4

SUCCESS
  File     : /kaggle/working/highlights_final.mp4
  Size     : 56.3 MB
  Runtime  : ~5m 0s  (10 × 30-sec clips)


In [13]:
# CELL 13 — Summary
print('=' * 60)
print('PIPELINE COMPLETE')
print('=' * 60)
print()
print('Files produced in /kaggle/working:')
for f in [CSV_30S, CSV_SORTED, CSV_GROUPED, CSV_TOP_CHRONO, FINAL_VIDEO]:
    if os.path.exists(f):
        size = os.path.getsize(f)
        unit = f'{size/1024:.1f} KB' if size < 1e6 else f'{size/1_048_576:.1f} MB'
        print(f'  {os.path.basename(f):<30}  {unit}')
print()
print('Top 10 moments included (chronological):')
for _, r in df_top_chrono.iterrows():
    print(f"  {r['selected_timestamp']}  →  intensity {r['selected_intensity']:.1f}%")

PIPELINE COMPLETE

Files produced in /kaggle/working:
  intensity_30s.csv               6.9 KB
  intensity_sorted.csv            6.9 KB
  peaks_grouped.csv               1.4 KB
  top10_chronological.csv         0.3 KB
  highlights_final.mp4            56.3 MB

Top 10 moments included (chronological):
  0:01:00  →  intensity 65.5%
  0:20:30  →  intensity 40.8%
  0:29:00  →  intensity 40.5%
  0:33:00  →  intensity 99.0%
  0:40:30  →  intensity 43.5%
  0:44:00  →  intensity 45.2%
  1:20:00  →  intensity 42.6%
  2:08:30  →  intensity 64.2%
  2:58:00  →  intensity 58.7%
  3:03:00  →  intensity 52.9%
